# Image Captioning - Training & Evaluation

This notebook trains and evaluates the model using the ViT + GPT-2 architecture.

In [ ]:
!pip install transformers -q

import os
import json
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from dataloader import get_dataloaders, get_test_dataloader
from model import ImageCaptioningModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
config = {
    'encoder_name': 'google/vit-base-patch16-224-in21k',
    'decoder_name': 'gpt2',
    'checkpoint_dir': 'checkpoints',
    'max_len': 128,
    'learning_rate': 5e-5,
    'weight_decay': 1e-4,
    'epochs': 20,
    'batch_size': 16,
    'patience': 5,
    'annotations_path': os.path.join('..', 'data', 'raw', 'tiny-trcap-en.json'),
    'image_dir': os.path.join('..', 'data', 'raw', 'images'),
    'split_ratios': (0.70, 0.10, 0.20),
    'caption_mode': 'random',
    'augment': True,
    'num_workers': 2,
    'seed': 42,
}

os.makedirs(config['checkpoint_dir'], exist_ok=True)

In [ ]:
train_loader, val_loader = get_dataloaders(config)
test_loader = get_test_dataloader(config)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

In [ ]:
model = ImageCaptioningModel(
    encoder_name=config['encoder_name'],
    decoder_name=config['decoder_name']
).to(device)

optimizer = AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=config['epochs'], eta_min=1e-6)

In [ ]:
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    total_steps = 0

    for batch in dataloader:
        if batch is None: continue
        images = batch['image'].to(device)
        captions = batch['caption'].to(device)

        labels = captions.clone()
        labels[labels == model.tokenizer.pad_token_id] = -100

        optimizer.zero_grad()
        outputs = model(pixel_values=images, labels=labels)
        loss = outputs.loss
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_steps += 1

    return total_loss / max(total_steps, 1)

def validate(model, dataloader, device):
    model.eval()
    total_loss = 0
    total_steps = 0
    with torch.no_grad():
        for batch in dataloader:
            if batch is None: continue
            images = batch['image'].to(device)
            captions = batch['caption'].to(device)
            
            labels = captions.clone()
            labels[labels == model.tokenizer.pad_token_id] = -100

            outputs = model(pixel_values=images, labels=labels)
            loss = outputs.loss

            total_loss += loss.item()
            total_steps += 1
            
    return total_loss / max(total_steps, 1)

In [ ]:
best_val_loss = float('inf')
patience_counter = 0
history = []

print("Starting training...")
for epoch in range(1, config['epochs'] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss = validate(model, val_loader, device)
    scheduler.step()

    print(f"Epoch {epoch}/{config['epochs']} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'config': config
        }, os.path.join(config['checkpoint_dir'], 'best_model.pt'))
        print("  -> Best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= config['patience']:
            print(f"Early stopping triggered after {epoch} epochs.")
            break

with open(os.path.join(config['checkpoint_dir'], 'history.json'), 'w') as f:
    json.dump(history, f, indent=2)

In [ ]:
# Evaluation on Test Set
print("\nLoading best model for evaluation...")
best_ckpt = torch.load(os.path.join(config['checkpoint_dir'], 'best_model.pt'), map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])

test_loss = validate(model, test_loader, device)
print(f"Test Loss: {test_loss:.4f}")